
## Ускоренный training 

Чтобы сделать воспроизведение выполнимым в Colab, используется одинаковый ускоренный режим для baseline и всех subset-моделей:

- ResNet-18 для CIFAR-10;
- batch size 512;
- SGD + momentum + weight decay;
- AMP/FP16 на CUDA;
- learning-rate levels:
  `0.4 → 0.08 → 0.016 → 0.0032 → 0.0008`;
- переход на следующий LR происходит по plateau среднего train loss либо по лимиту эпох;
- первый LR используется максимум 20 full-data-equivalent эпох, каждый следующий — максимум 15;
- на последней ступени plateau приводит к early stopping.


## Параллельный запуск в нескольких Colab

Проект может лежать в общей папке Google Drive. На каждом Colab достаточно изменить `RUN_METHOD`:

```python
RUN_METHOD = "el2n"
RUN_METHOD = "random"
RUN_METHOD = "forgetting"
RUN_METHOD = "grand"
```

Каждый Colab обучит только выбранный метод на **75%, 50% и 25%** данных и запишет результаты в отдельные директории. `RUN_METHOD = "all"` запускает все методы последовательно в одном ноутбуке.

Baseline и scoring artifacts общие для всех аккаунтов. Baseline пересчитывается только если `RECOMPUTE_BASELINE = True`; для параллельных subset-run обычно используется `False`.


In [ ]:
# Основные параметры эксперимента.
# DEBUG=True запускает короткую проверку пайплайна и пишет результаты в debug/, не смешивая их с production.
DEBUG = False

# Какой метод считает этот Colab: "el2n", "random", "forgetting", "grand" или "all".
RUN_METHOD = "el2n"

# True — заново обучить baseline на 100% CIFAR-10.
# False — использовать уже завершённый baseline из общей папки Drive.
# Для параллельных Colab после готовности baseline оставляется False.
RECOMPUTE_BASELINE = False

# Число независимых запусков для вычисления score и для финального retraining.
# В статье использовалось больше seed; здесь по умолчанию 1 из-за ограничений compute.
NUM_SCORING_SEEDS = 1
NUM_EVAL_SEEDS = 1

# Доля train-датасета, которую сохраняем после pruning.
RETAIN_FRACTIONS = [0.75, 0.50, 0.25]

# Общие директории протокола. Они оставлены совместимыми с уже посчитанными run.
EXPERIMENT_NAMESPACE = "fast_bs512_plateau_v2"
SUBSET_RUN_NAMESPACE = "subsets_lr0008_v3"


TRAIN_BATCH_SIZE = 512
TEST_BATCH_SIZE = 1024
LR_LEVELS = [0.4, 0.08, 0.016, 0.0032, 0.0008]
PLATEAU_PATIENCE_EPOCHS = 5
PLATEAU_MIN_DELTA = 0.002
FIRST_LR_MIN_EPOCHS = 10
FIRST_LR_MAX_EPOCHS = 20
LATER_LR_MIN_EPOCHS = 5
LATER_LR_MAX_EPOCHS = 15
MIN_TOTAL_EPOCHS = 20


FULL_SCORE_EPOCHS = [0, 1, 2, 5, 10, 20]
EL2N_SELECTION_EPOCH = 20
GRAND_SELECTION_EPOCH = 20
GRAND_ARCHIVE_DIR_NAME = "seed_0_before_recompute_20261009_080948"


USE_AMP = True
USE_CHANNELS_LAST = True
USE_LOCAL_DATA_CACHE = True
NUM_WORKERS = None
PREFETCH_FACTOR = 4
EVAL_EVERY_FRACTION = 0.05
CHECKPOINT_EVERY_FRACTION = 0.20
PROJECT_NAME = "data_diet_reproduction"

ALL_SELECTION_METHODS = ["el2n", "random", "forgetting", "grand"]

if RUN_METHOD == "all":
    ACTIVE_METHODS = list(ALL_SELECTION_METHODS)
elif RUN_METHOD in ALL_SELECTION_METHODS:
    ACTIVE_METHODS = [RUN_METHOD]
else:
    raise ValueError(f"Unknown RUN_METHOD: {RUN_METHOD}")

FULL_RUN_PLAN = [
    ("el2n", 0.75), ("random", 0.75), ("forgetting", 0.75), ("grand", 0.75),
    ("el2n", 0.50), ("random", 0.50), ("forgetting", 0.50), ("grand", 0.50),
    ("el2n", 0.25), ("random", 0.25), ("forgetting", 0.25), ("grand", 0.25),
]

ACTIVE_RUN_PLAN = [
    (method, fraction)
    for method, fraction in FULL_RUN_PLAN
    if method in ACTIVE_METHODS
]

DEBUG_FIRST_LR_MAX_EPOCHS = 3
DEBUG_LATER_LR_MAX_EPOCHS = 2
DEBUG_PLATEAU_PATIENCE_EPOCHS = 1

print("DEBUG:", DEBUG)
print("RUN_METHOD:", RUN_METHOD)
print("RECOMPUTE_BASELINE:", RECOMPUTE_BASELINE)
print("ACTIVE_RUN_PLAN:", ACTIVE_RUN_PLAN)
print("NUM_SCORING_SEEDS:", NUM_SCORING_SEEDS)
print("NUM_EVAL_SEEDS:", NUM_EVAL_SEEDS)


In [ ]:
import os
import sys
import gc
import json
import math
import time
import random
import platform
import shutil
from datetime import datetime
from contextlib import nullcontext
from pathlib import Path
from typing import Optional, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import Dataset, DataLoader, Subset

from scipy.stats import spearmanr
from tqdm.auto import tqdm

IN_COLAB = "google.colab" in sys.modules
MODE_NAME = "debug" if DEBUG else "production"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_BASE = Path("/content/drive/MyDrive") / PROJECT_NAME
else:
    PROJECT_BASE = Path.cwd() / PROJECT_NAME

PROJECT_ROOT = PROJECT_BASE / MODE_NAME / EXPERIMENT_NAMESPACE

DRIVE_DATA_DIR = PROJECT_BASE / "data"
if IN_COLAB and USE_LOCAL_DATA_CACHE:
    LOCAL_DATA_DIR = Path("/content") / f"{PROJECT_NAME}_data"
    DATA_DIR = LOCAL_DATA_DIR
else:
    LOCAL_DATA_DIR = None
    DATA_DIR = DRIVE_DATA_DIR

DIRS = {
    "root": PROJECT_ROOT,
    "data": DATA_DIR,
    "drive_data": DRIVE_DATA_DIR,
    "runs": PROJECT_ROOT / "runs",
    "scoring_runs": PROJECT_ROOT / "runs" / "scoring",
    "subset_runs": PROJECT_ROOT / "runs" / SUBSET_RUN_NAMESPACE,
    "scores": PROJECT_ROOT / "scores",
    "aggregated_scores": PROJECT_ROOT / "scores" / "aggregated",
    "subsets": PROJECT_ROOT / "subsets",
    "results": PROJECT_ROOT / "results",
    "figures": PROJECT_ROOT / "figures",
    "configs": PROJECT_ROOT / "configs",
}

for p in DIRS.values():
    Path(p).mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
else:
    print("WARNING: CUDA GPU not detected.")

if NUM_WORKERS is None:
    NUM_WORKERS = max(1, min(4, int(os.cpu_count() or 2)))

FULL_TRAIN_EXAMPLES = 50_000
FULL_STEPS_PER_EPOCH = FULL_TRAIN_EXAMPLES // TRAIN_BATCH_SIZE

if DEBUG:
    ACTIVE_FIRST_LR_MAX_EPOCHS = DEBUG_FIRST_LR_MAX_EPOCHS
    ACTIVE_LATER_LR_MAX_EPOCHS = DEBUG_LATER_LR_MAX_EPOCHS
    ACTIVE_PLATEAU_PATIENCE_EPOCHS = DEBUG_PLATEAU_PATIENCE_EPOCHS
    ACTIVE_MIN_TOTAL_EPOCHS = min(2, DEBUG_FIRST_LR_MAX_EPOCHS)
else:
    ACTIVE_FIRST_LR_MAX_EPOCHS = FIRST_LR_MAX_EPOCHS
    ACTIVE_LATER_LR_MAX_EPOCHS = LATER_LR_MAX_EPOCHS
    ACTIVE_PLATEAU_PATIENCE_EPOCHS = PLATEAU_PATIENCE_EPOCHS
    ACTIVE_MIN_TOTAL_EPOCHS = MIN_TOTAL_EPOCHS

TOTAL_EQUIV_EPOCHS = (
    ACTIVE_FIRST_LR_MAX_EPOCHS
    + (len(LR_LEVELS) - 1) * ACTIVE_LATER_LR_MAX_EPOCHS
)
TOTAL_STEPS = TOTAL_EQUIV_EPOCHS * FULL_STEPS_PER_EPOCH
SUBSET_TARGET_STEPS = TOTAL_STEPS

EVAL_EVERY_STEPS = max(
    FULL_STEPS_PER_EPOCH,
    int(round(TOTAL_STEPS * float(EVAL_EVERY_FRACTION))),
)
CHECKPOINT_EVERY_STEPS = max(
    FULL_STEPS_PER_EPOCH,
    int(round(TOTAL_STEPS * float(CHECKPOINT_EVERY_FRACTION))),
)

SCORE_EPOCHS = [e for e in FULL_SCORE_EPOCHS if e <= TOTAL_EQUIV_EPOCHS]
if DEBUG:
    EL2N_ACTIVE_SELECTION_EPOCH = max(SCORE_EPOCHS)
else:
    EL2N_ACTIVE_SELECTION_EPOCH = EL2N_SELECTION_EPOCH

GRAND_ACTIVE_SELECTION_EPOCH = GRAND_SELECTION_EPOCH

SCORING_SEEDS = list(range(NUM_SCORING_SEEDS))
EVAL_SEEDS = [10_000 + i for i in range(NUM_EVAL_SEEDS)]

CONFIG = {
    "debug": DEBUG,
    "experiment_namespace": EXPERIMENT_NAMESPACE,
    "run_method": RUN_METHOD,
    "active_methods": ACTIVE_METHODS,
    "recompute_baseline": RECOMPUTE_BASELINE,
    "subset_run_namespace": SUBSET_RUN_NAMESPACE,
    "num_scoring_seeds": NUM_SCORING_SEEDS,
    "num_eval_seeds": NUM_EVAL_SEEDS,
    "scoring_seeds": SCORING_SEEDS,
    "eval_seeds": EVAL_SEEDS,
    "retain_fractions": RETAIN_FRACTIONS,
    "full_run_plan": FULL_RUN_PLAN,
    "active_run_plan": ACTIVE_RUN_PLAN,
    "selection_methods": ALL_SELECTION_METHODS,
    "batch_size": TRAIN_BATCH_SIZE,
    "test_batch_size": TEST_BATCH_SIZE,
    "optimizer": "SGD",
    "momentum": 0.9,
    "weight_decay": 0.0005,
    "nesterov": True,
    "scheduler": "AdaptivePlateauStageLR",
    "lr_levels": LR_LEVELS,
    "base_lr": LR_LEVELS[0],
    "final_lr": LR_LEVELS[-1],
    "plateau_patience_epochs": ACTIVE_PLATEAU_PATIENCE_EPOCHS,
    "plateau_min_delta": PLATEAU_MIN_DELTA,
    "first_lr_min_epochs": 1 if DEBUG else FIRST_LR_MIN_EPOCHS,
    "first_lr_max_epochs": ACTIVE_FIRST_LR_MAX_EPOCHS,
    "later_lr_min_epochs": 1 if DEBUG else LATER_LR_MIN_EPOCHS,
    "later_lr_max_epochs": ACTIVE_LATER_LR_MAX_EPOCHS,
    "min_total_epochs": ACTIVE_MIN_TOTAL_EPOCHS,
    "full_steps_per_epoch": FULL_STEPS_PER_EPOCH,
    "total_equiv_epochs": TOTAL_EQUIV_EPOCHS,
    "total_steps": TOTAL_STEPS,
    "subset_target_steps": SUBSET_TARGET_STEPS,
    "score_epochs": SCORE_EPOCHS,
    "el2n_selection_epoch": EL2N_ACTIVE_SELECTION_EPOCH,
    "forgetting_selection_epoch": "final",
    "grand_selection_epoch": GRAND_ACTIVE_SELECTION_EPOCH,
    "grand_archive_dir_name": GRAND_ARCHIVE_DIR_NAME,
    "mode_name": MODE_NAME,
    "use_amp": USE_AMP,
    "use_channels_last": USE_CHANNELS_LAST,
    "use_local_data_cache": USE_LOCAL_DATA_CACHE,
    "num_workers": NUM_WORKERS,
    "prefetch_factor": PREFETCH_FACTOR,
    "eval_every_steps": EVAL_EVERY_STEPS,
    "checkpoint_every_steps": CHECKPOINT_EVERY_STEPS,
    "device": str(DEVICE),
}

CONFIG_PATH = DIRS["configs"] / f"config_{RUN_METHOD}.json"
with open(CONFIG_PATH, "w") as f:
    json.dump(CONFIG, f, indent=2)

PROTOCOL_CONFIG = {
    **CONFIG,
    "run_method": "all",
    "active_methods": ALL_SELECTION_METHODS,
    "active_run_plan": FULL_RUN_PLAN,
}
with open(DIRS["configs"] / "config_protocol.json", "w") as f:
    json.dump(PROTOCOL_CONFIG, f, indent=2)

ENVIRONMENT = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "device": str(DEVICE),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "cpu_count": os.cpu_count(),
}
with open(DIRS["configs"] / "environment.json", "w") as f:
    json.dump(ENVIRONMENT, f, indent=2)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("DEVICE:", DEVICE)
print("FULL_STEPS_PER_EPOCH:", FULL_STEPS_PER_EPOCH)
print("MAX_STEPS_PER_RUN:", TOTAL_STEPS)
print("LR_LEVELS:", LR_LEVELS)
print("SUBSET_RUNS:", len(ACTIVE_RUN_PLAN) * len(EVAL_SEEDS))


## 2. Данные и воспроизводимость

CIFAR-10 кэшируется локально в `/content`, а постоянная копия и результаты хранятся на Google Drive.


In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)
CIFAR10_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def seed_worker(worker_id: int):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

TRAIN_TRANSFORM = transforms.Compose([
    transforms.RandomCrop(32, padding=4, padding_mode="reflect"),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

EVAL_TRANSFORM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

class IndexedSortedCIFAR10(Dataset):
    def __init__(self, root: Path, train: bool, transform, download: bool = True):
        self.base = CIFAR10(
            root=str(root), train=train, download=download, transform=None
        )
        targets = np.asarray(self.base.targets)
        self.order = np.argsort(targets, kind="stable")
        self.transform = transform

    def __len__(self):
        return len(self.order)

    def __getitem__(self, idx):
        base_idx = int(self.order[idx])
        image, target = self.base[base_idx]
        if self.transform is not None:
            image = self.transform(image)
        return image, int(target), int(idx)

def _has_valid_cifar(path: Path) -> bool:
    path = Path(path)
    try:
        CIFAR10(root=str(path), train=True, download=False)
        CIFAR10(root=str(path), train=False, download=False)
        return True
    except Exception:
        return False

def prepare_local_data_cache():
    if not (IN_COLAB and USE_LOCAL_DATA_CACHE):
        return

    DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)

    if _has_valid_cifar(LOCAL_DATA_DIR):
        print("[data] valid local /content CIFAR cache already present:", LOCAL_DATA_DIR)
        return

    if LOCAL_DATA_DIR.exists():
        print("[data] removing incomplete local CIFAR cache:", LOCAL_DATA_DIR)
        shutil.rmtree(LOCAL_DATA_DIR)
    LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

    if _has_valid_cifar(DRIVE_DATA_DIR):
        print("[data] copying verified CIFAR-10 from Drive to /content ...")
        shutil.copytree(
            DRIVE_DATA_DIR,
            LOCAL_DATA_DIR,
            dirs_exist_ok=True,
        )
        print("[data] local cache ready:", LOCAL_DATA_DIR)
    else:
        print(
            "[data] no valid persistent CIFAR cache found; "
            "torchvision will download it once into /content."
        )

prepare_local_data_cache()

def make_datasets():
    train_aug = IndexedSortedCIFAR10(
        DIRS["data"], train=True, transform=TRAIN_TRANSFORM, download=True
    )
    train_eval = IndexedSortedCIFAR10(
        DIRS["data"], train=True, transform=EVAL_TRANSFORM, download=True
    )
    test_eval = IndexedSortedCIFAR10(
        DIRS["data"], train=False, transform=EVAL_TRANSFORM, download=True
    )

    if IN_COLAB and USE_LOCAL_DATA_CACHE and not _has_valid_cifar(DRIVE_DATA_DIR):
        print("[data] persisting CIFAR-10 cache back to Drive once...")
        shutil.copytree(
            LOCAL_DATA_DIR,
            DRIVE_DATA_DIR,
            dirs_exist_ok=True,
        )

    return train_aug, train_eval, test_eval

TRAIN_AUG_DATASET, TRAIN_EVAL_DATASET, TEST_DATASET = make_datasets()

print("Train:", len(TRAIN_AUG_DATASET))
print("Test :", len(TEST_DATASET))
print("Training reads from:", DIRS["data"])


In [ ]:
raw_train = IndexedSortedCIFAR10(
    DIRS["data"], train=True, transform=None, download=True
)

fig = plt.figure(figsize=(10, 5))
for i in range(10):
    ax = fig.add_subplot(2, 5, i + 1)
    img, label, idx = raw_train[i * 5000]
    ax.imshow(img)
    ax.set_title(f"{CIFAR10_CLASSES[label]}\nidx={idx}")
    ax.axis("off")
fig.tight_layout()
plt.show()


## 3. Модель

Используется CIFAR-версия ResNet-18: 3×3 первый convolution, stride 1, без initial max-pooling.


In [ ]:
def create_resnet18_cifar(num_classes: int = 10) -> nn.Module:
    model = torchvision.models.resnet18(weights=None, num_classes=num_classes)
    model.conv1 = nn.Conv2d(
        3, 64, kernel_size=3, stride=1, padding=1, bias=False
    )
    model.maxpool = nn.Identity()
    return model

_tmp_model = create_resnet18_cifar()
print(f"Parameters: {sum(p.numel() for p in _tmp_model.parameters()):,}")
del _tmp_model


## 4. Обучение

Все модели используют один и тот же адаптивный пятиступенчатый LR schedule и могут безопасно продолжаться после обрыва Colab.


In [ ]:
def atomic_json_dump(obj, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)

def append_metrics_row(path: Path, row: dict):
    path = Path(path)
    df = pd.DataFrame([row])
    if path.exists():
        df.to_csv(path, mode="a", header=False, index=False)
    else:
        df.to_csv(path, index=False)

def update_live_results(summary: dict):
    live_tag = RUN_METHOD if RUN_METHOD != "all" else "all"
    path = DIRS["results"] / f"live_results_{live_tag}.csv"
    row = pd.DataFrame([summary])
    key_cols = ["method", "fraction", "seed"]

    if path.exists():
        old = pd.read_csv(path)
        combined = pd.concat([old, row], ignore_index=True, sort=False)
        if all(c in combined.columns for c in key_cols):
            combined = combined.drop_duplicates(subset=key_cols, keep="last")
    else:
        combined = row

    if all(c in combined.columns for c in key_cols):
        combined = combined.sort_values(key_cols).reset_index(drop=True)
    combined.to_csv(path, index=False)
    print("[live results]", path)
    display(combined)

def run_is_complete(
    run_dir: Path,
    target_steps: Optional[int] = None,
) -> bool:
    summary_path = Path(run_dir) / "summary.json"
    if not summary_path.exists():
        return False
    try:
        with open(summary_path) as f:
            summary = json.load(f)
        if not bool(summary.get("complete", False)):
            return False
        if target_steps is None:
            return True
        return int(summary.get("total_steps", 0)) >= int(target_steps)
    except Exception:
        return False

def archive_existing_path(path: Path, label: str = "archive"):
    path = Path(path)
    if not path.exists():
        return None
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    archived = path.with_name(f"{path.name}_{label}_{stamp}")
    counter = 1
    while archived.exists():
        archived = path.with_name(f"{path.name}_{label}_{stamp}_{counter}")
        counter += 1
    path.rename(archived)
    print("[archive]", path, "->", archived)
    return archived

def capture_rng_state():
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state

def restore_rng_state(state):
    if not state:
        return
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if torch.cuda.is_available() and "cuda" in state:
        torch.cuda.set_rng_state_all(state["cuda"])

def make_loader(
    dataset,
    batch_size: int,
    shuffle: bool,
    drop_last: bool,
    seed: int,
):
    generator = torch.Generator()
    generator.manual_seed(seed)

    kwargs = dict(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        worker_init_fn=seed_worker,
        generator=generator,
        persistent_workers=(NUM_WORKERS > 0),
    )
    if NUM_WORKERS > 0:
        kwargs["prefetch_factor"] = PREFETCH_FACTOR

    return DataLoader(**kwargs)

@torch.no_grad()
def evaluate(model: nn.Module, dataset, batch_size: int = 1024):
    model.eval()
    loader = make_loader(
        dataset,
        batch_size,
        shuffle=False,
        drop_last=False,
        seed=12345,
    )

    total_loss = 0.0
    total_correct = 0
    total = 0

    for x, y, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        if USE_CHANNELS_LAST and DEVICE.type == "cuda":
            x = x.contiguous(memory_format=torch.channels_last)
        y = y.to(DEVICE, non_blocking=True)

        with _autocast_context():
            logits = model(x)
            batch_loss = F.cross_entropy(
                logits, y, reduction="sum"
            )

        total_loss += float(batch_loss.item())
        total_correct += int(
            (logits.argmax(dim=1) == y).sum().item()
        )
        total += y.numel()

    return {
        "loss": total_loss / total,
        "accuracy": total_correct / total,
    }

def torch_load_compat(path: Path):
    try:
        return torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location="cpu",
        )

def save_model_only(
    model: nn.Module,
    path: Path,
    step: int,
    equiv_epoch: int,
    seed: int,
):
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "model_state": {
            k: v.detach().cpu()
            for k, v in model.state_dict().items()
        },
        "step": int(step),
        "equiv_epoch": int(equiv_epoch),
        "seed": int(seed),
    }
    torch.save(payload, path)

def load_model_from_checkpoint(path: Path):
    payload = torch_load_compat(path)
    model = create_resnet18_cifar()
    model.load_state_dict(payload["model_state"])
    model.to(DEVICE)
    if USE_CHANNELS_LAST and DEVICE.type == "cuda":
        model = model.to(memory_format=torch.channels_last)
    return model, payload

def subset_dataset(dataset, indices: Optional[np.ndarray]):
    if indices is None:
        return dataset
    return Subset(dataset, [int(i) for i in indices])

def _make_grad_scaler():
    enabled = USE_AMP and torch.cuda.is_available()
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=enabled)

def _autocast_context():
    enabled = USE_AMP and torch.cuda.is_available()
    if not enabled:
        return nullcontext()
    try:
        return torch.amp.autocast("cuda", enabled=True)
    except Exception:
        return torch.cuda.amp.autocast(enabled=True)

class AdaptivePlateauStageLR:
    def __init__(
        self,
        optimizer,
        *,
        lr_levels,
        patience_epochs: int,
        min_delta: float,
        first_min_epochs: int,
        first_max_epochs: int,
        later_min_epochs: int,
        later_max_epochs: int,
        min_total_epochs: int,
    ):
        self.optimizer = optimizer
        self.lr_levels = [float(x) for x in lr_levels]
        if len(self.lr_levels) < 1:
            raise ValueError("lr_levels must not be empty")
        self.patience_epochs = int(patience_epochs)
        self.min_delta = float(min_delta)
        self.first_min_epochs = int(first_min_epochs)
        self.first_max_epochs = int(first_max_epochs)
        self.later_min_epochs = int(later_min_epochs)
        self.later_max_epochs = int(later_max_epochs)
        self.min_total_epochs = int(min_total_epochs)

        self.stage = 0
        self.epochs_in_stage = 0
        self.best_metric = float("inf")
        self.bad_epochs = 0
        self.last_reason = "init"
        self._set_lr(self.lr_levels[0])

    def _set_lr(self, lr: float):
        for group in self.optimizer.param_groups:
            group["lr"] = float(lr)

    @property
    def current_lr(self) -> float:
        return float(self.optimizer.param_groups[0]["lr"])

    @property
    def final_stage(self) -> int:
        return len(self.lr_levels) - 1

    def _limits(self):
        if self.stage == 0:
            return self.first_min_epochs, self.first_max_epochs
        return self.later_min_epochs, self.later_max_epochs

    def move_to_stage(self, stage: int, reason: str = "manual"):
        stage = int(stage)
        if not (0 <= stage <= self.final_stage):
            raise ValueError(f"Invalid LR stage {stage}")
        self.stage = stage
        self.epochs_in_stage = 0
        self.best_metric = float("inf")
        self.bad_epochs = 0
        self.last_reason = reason
        self._set_lr(self.lr_levels[stage])

    def step_epoch(self, metric: float, global_epoch: int):
        metric = float(metric)
        self.epochs_in_stage += 1

        improved = metric < (self.best_metric - self.min_delta)
        if improved:
            self.best_metric = metric
            self.bad_epochs = 0
        else:
            self.bad_epochs += 1

        min_epochs, max_epochs = self._limits()
        plateau = (
            self.epochs_in_stage >= min_epochs
            and self.bad_epochs >= self.patience_epochs
        )
        stage_cap = self.epochs_in_stage >= max_epochs

        reason = "plateau" if plateau else ("stage_cap" if stage_cap else None)

        event = {
            "changed": False,
            "stop": False,
            "reason": None,
            "stage": self.stage,
            "epoch": int(global_epoch),
            "metric": metric,
            "lr_before": self.current_lr,
            "lr_after": self.current_lr,
            "epochs_in_stage": self.epochs_in_stage,
            "bad_epochs": self.bad_epochs,
        }

        if reason is None:
            return event

        if self.stage >= self.final_stage:
            if global_epoch >= self.min_total_epochs:
                event.update({"stop": True, "reason": f"final_{reason}"})
                self.last_reason = event["reason"]
            return event

        old_lr = self.current_lr
        self.stage += 1
        new_lr = self.lr_levels[self.stage]
        self._set_lr(new_lr)
        self.epochs_in_stage = 0
        self.best_metric = float("inf")
        self.bad_epochs = 0
        self.last_reason = reason

        event.update({
            "changed": True,
            "reason": reason,
            "stage": self.stage,
            "lr_before": old_lr,
            "lr_after": new_lr,
        })
        return event

    def state_dict(self):
        return {
            "stage": self.stage,
            "epochs_in_stage": self.epochs_in_stage,
            "best_metric": self.best_metric,
            "bad_epochs": self.bad_epochs,
            "last_reason": self.last_reason,
            "current_lr": self.current_lr,
            "lr_levels": self.lr_levels,
        }

    def load_state_dict(self, state):
        self.stage = int(state.get("stage", 0))
        self.epochs_in_stage = int(state.get("epochs_in_stage", 0))
        self.best_metric = float(state.get("best_metric", float("inf")))
        self.bad_epochs = int(state.get("bad_epochs", 0))
        self.last_reason = state.get("last_reason", "resume")
        saved_lr = float(state.get("current_lr", self.lr_levels[min(self.stage, self.final_stage)]))
        self._set_lr(saved_lr)

def _save_training_checkpoint(
    *,
    model,
    optimizer,
    scheduler,
    scaler,
    step,
    best_test_accuracy,
    latest_path,
    track_forgetting,
    forget_counts,
    prev_correct,
    ever_correct,
    epoch_loss_sum,
    epoch_examples,
):
    payload = {
        "model_state": {
            k: v.detach().cpu()
            for k, v in model.state_dict().items()
        },
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state": scaler.state_dict(),
        "step": int(step),
        "best_test_accuracy": float(best_test_accuracy),
        "rng_state": capture_rng_state(),
        "protocol": SUBSET_RUN_NAMESPACE,
        "epoch_loss_sum": float(epoch_loss_sum),
        "epoch_examples": int(epoch_examples),
    }
    if track_forgetting:
        payload.update({
            "forget_counts": forget_counts,
            "prev_correct": prev_correct,
            "ever_correct": ever_correct,
        })
    torch.save(payload, latest_path)

def train_fixed_steps(
    *,
    run_dir: Path,
    train_dataset,
    test_dataset,
    seed: int,
    method: str,
    fraction: float,
    subset_indices: Optional[np.ndarray] = None,
    track_forgetting: bool = False,
    save_score_checkpoints: bool = False,
    target_steps: Optional[int] = None,
):
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "models").mkdir(exist_ok=True)
    (run_dir / "forgetting").mkdir(exist_ok=True)

    total_steps = int(
        CONFIG["total_steps"] if target_steps is None else target_steps
    )
    if total_steps <= 0 or total_steps > int(CONFIG["total_steps"]):
        raise ValueError(
            f"target_steps={total_steps} must be in [1, {CONFIG['total_steps']}]"
        )

    if run_is_complete(run_dir, None):
        with open(run_dir / "summary.json") as f:
            existing_summary = json.load(f)
        print(f"[skip complete] {run_dir}")
        return existing_summary

    set_seed(seed)

    selected_dataset = subset_dataset(
        train_dataset, subset_indices
    )
    if len(selected_dataset) < CONFIG["batch_size"]:
        raise ValueError("Subset is smaller than training batch size.")

    run_config = {
        **CONFIG,
        "seed": seed,
        "method": method,
        "fraction": fraction,
        "subset_size": len(selected_dataset),
        "run_dir": str(run_dir),
        "track_forgetting": track_forgetting,
        "save_score_checkpoints": save_score_checkpoints,
        "target_steps": total_steps,
        "target_equiv_epochs": total_steps / FULL_STEPS_PER_EPOCH,
        "accelerated_replication": True,
    }
    atomic_json_dump(run_config, run_dir / "config.json")

    model = create_resnet18_cifar().to(DEVICE)
    if USE_CHANNELS_LAST and DEVICE.type == "cuda":
        model = model.to(memory_format=torch.channels_last)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=CONFIG["lr_levels"][0],
        momentum=CONFIG["momentum"],
        weight_decay=CONFIG["weight_decay"],
        nesterov=CONFIG["nesterov"],
    )
    scheduler = AdaptivePlateauStageLR(
        optimizer,
        lr_levels=CONFIG["lr_levels"],
        patience_epochs=CONFIG["plateau_patience_epochs"],
        min_delta=CONFIG["plateau_min_delta"],
        first_min_epochs=CONFIG["first_lr_min_epochs"],
        first_max_epochs=CONFIG["first_lr_max_epochs"],
        later_min_epochs=CONFIG["later_lr_min_epochs"],
        later_max_epochs=CONFIG["later_lr_max_epochs"],
        min_total_epochs=CONFIG["min_total_epochs"],
    )
    scaler = _make_grad_scaler()

    start_step = 0
    best_test_accuracy = -float("inf")
    epoch_loss_sum = 0.0
    epoch_examples = 0

    forget_counts = np.zeros(
        len(TRAIN_AUG_DATASET), dtype=np.float32
    )
    prev_correct = np.zeros(
        len(TRAIN_AUG_DATASET), dtype=np.bool_
    )
    ever_correct = np.zeros(
        len(TRAIN_AUG_DATASET), dtype=np.bool_
    )

    latest_path = run_dir / "latest.pt"
    if latest_path.exists():
        ckpt = torch_load_compat(latest_path)

        ckpt_protocol = ckpt.get("protocol")
        if method != "full" and ckpt_protocol != SUBSET_RUN_NAMESPACE:
            raise RuntimeError(
                "Checkpoint protocol mismatch. Refusing to resume an old "
                f"subset run: {latest_path}. Expected {SUBSET_RUN_NAMESPACE!r}, "
                f"found {ckpt_protocol!r}."
            )

        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        if "scheduler_state" not in ckpt:
            raise RuntimeError(
                "This checkpoint has no adaptive scheduler state. "
                "Refusing to resume an incompatible run."
            )
        scheduler.load_state_dict(ckpt["scheduler_state"])

        if "scaler_state" in ckpt:
            scaler.load_state_dict(ckpt["scaler_state"])
        start_step = int(ckpt["step"])
        best_test_accuracy = float(
            ckpt.get("best_test_accuracy", best_test_accuracy)
        )
        if track_forgetting:
            forget_counts = ckpt["forget_counts"]
            prev_correct = ckpt["prev_correct"]
            ever_correct = ckpt["ever_correct"]
        epoch_loss_sum = float(ckpt.get("epoch_loss_sum", 0.0))
        epoch_examples = int(ckpt.get("epoch_examples", 0))
        restore_rng_state(ckpt.get("rng_state"))
        print(
            f"[resume] {run_dir.name}: "
            f"step {start_step}/{total_steps}"
        )

        metrics_path = run_dir / "metrics.csv"
        if metrics_path.exists():
            _metrics = pd.read_csv(metrics_path)
            _metrics = (
                _metrics[_metrics["step"] <= start_step]
                .drop_duplicates(subset=["step"], keep="last")
                .sort_values("step")
            )
            _metrics.to_csv(metrics_path, index=False)

    if save_score_checkpoints and 0 in SCORE_EPOCHS:
        init_path = run_dir / "models" / "epoch_000.pt"
        if not init_path.exists():
            if start_step == 0:
                save_model_only(
                    model, init_path, 0, 0, seed
                )
            else:
                raise FileNotFoundError(
                    "Initialization checkpoint is missing but the scoring run "
                    "already progressed."
                )

    loader = make_loader(
        selected_dataset,
        CONFIG["batch_size"],
        shuffle=True,
        drop_last=True,
        seed=seed + 17,
    )
    loader_iter = iter(loader)

    window_loss = 0.0
    window_correct = 0
    window_examples = 0
    window_start_time = time.time()

    score_steps = {
        int(epoch * FULL_STEPS_PER_EPOCH): int(epoch)
        for epoch in SCORE_EPOCHS
        if epoch > 0
    }
    forgetting_snapshot_steps = {
        int(epoch * FULL_STEPS_PER_EPOCH): int(epoch)
        for epoch in sorted(set(SCORE_EPOCHS))
        if epoch > 0
    }

    last_test_metrics = None
    stop_reason = "hard_cap"
    actual_step = start_step
    lr_events_path = run_dir / "lr_events.csv"

    pbar = tqdm(
        range(start_step + 1, total_steps + 1),
        desc=f"{method} {fraction:.0%} seed={seed}",
    )

    for step in pbar:
        try:
            x, y, idx = next(loader_iter)
        except StopIteration:
            loader_iter = iter(loader)
            x, y, idx = next(loader_iter)

        x = x.to(DEVICE, non_blocking=True)
        if USE_CHANNELS_LAST and DEVICE.type == "cuda":
            x = x.contiguous(memory_format=torch.channels_last)
        y = y.to(DEVICE, non_blocking=True)
        idx_np = idx.cpu().numpy()

        lr = float(optimizer.param_groups[0]["lr"])

        model.train()
        optimizer.zero_grad(set_to_none=True)

        with _autocast_context():
            logits = model(x)
            loss = F.cross_entropy(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        lr = float(optimizer.param_groups[0]["lr"])

        preds = logits.detach().argmax(dim=1)
        correct = (
            (preds == y)
            .detach()
            .cpu()
            .numpy()
            .astype(bool)
        )

        if track_forgetting:
            forgotten_now = prev_correct[idx_np] & (~correct)
            forget_counts[idx_np[forgotten_now]] += 1.0
            prev_correct[idx_np] = correct
            ever_correct[idx_np[correct]] = True

        batch_n = y.numel()
        window_loss += float(loss.item()) * batch_n
        window_correct += int(correct.sum())
        window_examples += batch_n
        epoch_loss_sum += float(loss.item()) * batch_n
        epoch_examples += batch_n
        actual_step = step

        if save_score_checkpoints and step in score_steps:
            equiv_epoch = score_steps[step]
            model_path = (
                run_dir / "models" / f"epoch_{equiv_epoch:03d}.pt"
            )
            if not model_path.exists():
                save_model_only(
                    model,
                    model_path,
                    step,
                    equiv_epoch,
                    seed,
                )

        if track_forgetting and step in forgetting_snapshot_steps:
            equiv_epoch = forgetting_snapshot_steps[step]
            forget_scores = forget_counts.copy()
            forget_scores[~ever_correct] = np.inf
            np.save(
                run_dir / "forgetting" / f"epoch_{equiv_epoch:03d}.npy",
                forget_scores,
            )

        scheduler_event = None
        if step % FULL_STEPS_PER_EPOCH == 0:
            global_epoch = step // FULL_STEPS_PER_EPOCH
            epoch_train_loss = epoch_loss_sum / max(epoch_examples, 1)
            scheduler_event = scheduler.step_epoch(
                epoch_train_loss,
                global_epoch,
            )
            lr = float(optimizer.param_groups[0]["lr"])

            if scheduler_event["changed"] or scheduler_event["stop"]:
                append_metrics_row(
                    lr_events_path,
                    {
                        "epoch": global_epoch,
                        "step": step,
                        "train_loss": epoch_train_loss,
                        "reason": scheduler_event["reason"],
                        "stage": scheduler_event["stage"],
                        "lr_before": scheduler_event["lr_before"],
                        "lr_after": scheduler_event["lr_after"],
                    },
                )
                print(
                    f"[lr event] epoch={global_epoch} "
                    f"reason={scheduler_event['reason']} "
                    f"lr {scheduler_event['lr_before']:.5g} -> "
                    f"{scheduler_event['lr_after']:.5g}"
                )

            epoch_loss_sum = 0.0
            epoch_examples = 0

        event_eval = bool(
            scheduler_event
            and (scheduler_event["changed"] or scheduler_event["stop"])
        )
        should_eval = (
            step % EVAL_EVERY_STEPS == 0
            or step == total_steps
            or event_eval
        )
        should_checkpoint = (
            step % CHECKPOINT_EVERY_STEPS == 0
            or step == total_steps
            or event_eval
        )

        if should_eval:
            test_metrics = evaluate(
                model,
                test_dataset,
                CONFIG["test_batch_size"],
            )
            last_test_metrics = test_metrics
            best_test_accuracy = max(
                best_test_accuracy,
                test_metrics["accuracy"],
            )

            elapsed = time.time() - window_start_time
            metrics_row = {
                "step": step,
                "equiv_epoch": step / FULL_STEPS_PER_EPOCH,
                "progress_fraction": step / total_steps,
                "method": method,
                "fraction": fraction,
                "seed": seed,
                "subset_size": len(selected_dataset),
                "train_loss": window_loss / max(window_examples, 1),
                "train_accuracy": window_correct / max(window_examples, 1),
                "test_loss": test_metrics["loss"],
                "test_accuracy": test_metrics["accuracy"],
                "learning_rate": lr,
                "lr_stage": scheduler.stage,
                "window_seconds": elapsed,
                "best_test_accuracy": best_test_accuracy,
            }
            append_metrics_row(
                run_dir / "metrics.csv",
                metrics_row,
            )

            window_loss = 0.0
            window_correct = 0
            window_examples = 0
            window_start_time = time.time()

            pbar.set_postfix(
                test_acc=f"{test_metrics['accuracy']:.4f}",
                lr=f"{lr:.4g}",
            )

        if should_checkpoint:
            _save_training_checkpoint(
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                scaler=scaler,
                step=step,
                best_test_accuracy=best_test_accuracy,
                latest_path=latest_path,
                track_forgetting=track_forgetting,
                forget_counts=forget_counts,
                prev_correct=prev_correct,
                ever_correct=ever_correct,
                epoch_loss_sum=epoch_loss_sum,
                epoch_examples=epoch_examples,
            )

        if scheduler_event and scheduler_event["stop"]:
            stop_reason = scheduler_event["reason"]
            print(
                f"[early stop] epoch={step / FULL_STEPS_PER_EPOCH:.1f}, "
                f"step={step:,}, reason={stop_reason}, lr={lr:.5g}"
            )
            break

    if track_forgetting:
        forget_scores = forget_counts.copy()
        forget_scores[~ever_correct] = np.inf
        np.save(run_dir / "forgetting" / "final.npy", forget_scores)

    if last_test_metrics is None:
        last_test_metrics = evaluate(
            model,
            test_dataset,
            CONFIG["test_batch_size"],
        )
        best_test_accuracy = max(
            best_test_accuracy,
            last_test_metrics["accuracy"],
        )

    metrics_path = run_dir / "metrics.csv"
    logged_runtime_seconds = None
    if metrics_path.exists():
        _runtime_metrics = pd.read_csv(metrics_path)
        if "window_seconds" in _runtime_metrics.columns:
            logged_runtime_seconds = float(
                _runtime_metrics["window_seconds"].sum()
            )

    summary = {
        "complete": True,
        "protocol": EXPERIMENT_NAMESPACE,
        "accelerated_replication": True,
        "batch_size": CONFIG["batch_size"],
        "scheduler": CONFIG["scheduler"],
        "base_lr": CONFIG["lr_levels"][0],
        "lr_levels": CONFIG["lr_levels"],
        "plateau_patience_epochs": CONFIG["plateau_patience_epochs"],
        "later_lr_max_epochs": CONFIG["later_lr_max_epochs"],
        "logged_runtime_seconds": logged_runtime_seconds,
        "logged_runtime_minutes": (
            None if logged_runtime_seconds is None else logged_runtime_seconds / 60.0
        ),
        "method": method,
        "fraction": fraction,
        "seed": seed,
        "subset_size": len(selected_dataset),
        "total_steps": int(actual_step),
        "max_steps": int(total_steps),
        "actual_equiv_epochs": float(actual_step / FULL_STEPS_PER_EPOCH),
        "lr_reductions": int(scheduler.stage),
        "final_lr": float(optimizer.param_groups[0]["lr"]),
        "stop_reason": stop_reason,
        "final_test_loss": last_test_metrics["loss"],
        "final_test_accuracy": last_test_metrics["accuracy"],
        "best_test_accuracy": best_test_accuracy,
        "run_dir": str(run_dir),
    }
    atomic_json_dump(
        summary,
        run_dir / "summary.json",
    )

    del model, optimizer, scheduler, scaler, loader, loader_iter
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary


## 5. Baseline на 100% данных

При `RECOMPUTE_BASELINE=True` baseline обучается с нуля. При `False` используется уже завершённый общий baseline.


In [ ]:
scoring_summaries = []

for seed in SCORING_SEEDS:
    run_dir = DIRS["scoring_runs"] / f"seed_{seed}"

    if RECOMPUTE_BASELINE:
        if run_dir.exists():
            archive_existing_path(run_dir, label="before_recompute")

        score_dir = DIRS["scores"] / "el2n" / f"seed_{seed}"
        if score_dir.exists():
            archive_existing_path(score_dir, label="before_recompute")

        if seed == SCORING_SEEDS[0]:
            if DIRS["aggregated_scores"].exists() and any(DIRS["aggregated_scores"].iterdir()):
                archive_existing_path(DIRS["aggregated_scores"], label="before_recompute")
                DIRS["aggregated_scores"].mkdir(parents=True, exist_ok=True)
            if DIRS["subsets"].exists() and any(DIRS["subsets"].iterdir()):
                archive_existing_path(DIRS["subsets"], label="before_recompute")
                DIRS["subsets"].mkdir(parents=True, exist_ok=True)
            if DIRS["subset_runs"].exists() and any(DIRS["subset_runs"].iterdir()):
                archive_existing_path(DIRS["subset_runs"], label="before_recompute")
                DIRS["subset_runs"].mkdir(parents=True, exist_ok=True)

        summary = train_fixed_steps(
            run_dir=run_dir,
            train_dataset=TRAIN_AUG_DATASET,
            test_dataset=TEST_DATASET,
            seed=seed,
            method="full",
            fraction=1.0,
            subset_indices=None,
            track_forgetting=True,
            save_score_checkpoints=True,
            target_steps=TOTAL_STEPS,
        )
    else:
        summary_path = run_dir / "summary.json"
        if not summary_path.exists():
            raise FileNotFoundError(
                f"Baseline not found: {summary_path}. "
                "Set RECOMPUTE_BASELINE=True on one Colab first."
            )
        with open(summary_path) as f:
            summary = json.load(f)

        if not bool(summary.get("complete", False)):
            raise RuntimeError(f"Baseline is incomplete: {run_dir}")

        if not math.isclose(
            float(summary.get("final_lr", float("nan"))),
            float(LR_LEVELS[-1]),
            rel_tol=0.0,
            abs_tol=1e-12,
        ):
            raise RuntimeError(
                f"Baseline final_lr={summary.get('final_lr')} does not match "
                f"the protocol final LR {LR_LEVELS[-1]}."
            )

        print(
            f"[reuse baseline] seed={seed} "
            f"test_acc={summary.get('final_test_accuracy', float('nan')):.4f}"
        )

    scoring_summaries.append(summary)
    update_live_results(summary)

pd.DataFrame(scoring_summaries)


## 6. Scoring

EL2N вычисляется по checkpoint ранней модели; forgetting собирается в ходе baseline; GraNd@20 загружается из заранее рассчитанных scores.


In [ ]:
def score_checkpoint_path(seed: int, epoch: int) -> Path:
    return (
        DIRS["scoring_runs"]
        / f"seed_{seed}"
        / "models"
        / f"epoch_{epoch:03d}.pt"
    )

def el2n_output_path(seed: int, epoch: int) -> Path:
    p = DIRS["scores"] / "el2n" / f"seed_{seed}"
    p.mkdir(parents=True, exist_ok=True)
    return p / f"epoch_{epoch:03d}.npz"

@torch.no_grad()
def compute_el2n(model: nn.Module, dataset, batch_size: int = 512):
    model.eval()
    loader = make_loader(
        dataset,
        batch_size,
        shuffle=False,
        drop_last=False,
        seed=23456,
    )

    all_idx = []
    all_el2n = []

    for x, y, idx in tqdm(loader, desc="EL2N", leave=False):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        with _autocast_context():
            logits = model(x)

        probs = torch.softmax(logits.float(), dim=1)
        onehot = F.one_hot(y, num_classes=10).float()
        el2n = torch.linalg.vector_norm(probs - onehot, ord=2, dim=1)

        all_idx.append(idx.numpy())
        all_el2n.append(el2n.cpu().numpy().astype(np.float32))

    return {
        "indices": np.concatenate(all_idx),
        "el2n": np.concatenate(all_el2n),
    }

if "el2n" in ACTIVE_METHODS:
    for seed in SCORING_SEEDS:
        for epoch in SCORE_EPOCHS:
            out_path = el2n_output_path(seed, epoch)

            if out_path.exists():
                print("[skip EL2N]", out_path)
                continue

            ckpt_path = score_checkpoint_path(seed, epoch)
            if not ckpt_path.exists():
                raise FileNotFoundError(f"Missing checkpoint: {ckpt_path}")

            model, _ = load_model_from_checkpoint(ckpt_path)
            scored = compute_el2n(model, TRAIN_EVAL_DATASET)
            np.savez_compressed(out_path, **scored)

            del model, scored
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

print("Score computation complete.")


## 7. Усреднение scores

При нескольких scoring seed EL2N и forgetting усредняются перед построением subset.


In [ ]:
def load_el2n_array(seed: int, epoch: int):
    d = np.load(el2n_output_path(seed, epoch))
    idx = d["indices"].astype(np.int64)
    scores = d["el2n"].astype(np.float32)

    full = np.full(len(TRAIN_EVAL_DATASET), np.nan, dtype=np.float32)
    full[idx] = scores
    return full

def aggregate_dense(arrays: List[np.ndarray]):
    return np.mean(np.stack(arrays, axis=0), axis=0)

if "el2n" in ACTIVE_METHODS:
    for epoch in SCORE_EPOCHS:
        arrays = [
            load_el2n_array(seed, epoch)
            for seed in SCORING_SEEDS
        ]
        np.save(
            DIRS["aggregated_scores"] / f"el2n_epoch_{epoch:03d}.npy",
            aggregate_dense(arrays),
        )

if "forgetting" in ACTIVE_METHODS:
    paths = [
        DIRS["scoring_runs"]
        / f"seed_{seed}"
        / "forgetting"
        / "final.npy"
        for seed in SCORING_SEEDS
    ]

    if not all(p.exists() for p in paths):
        missing = [str(p) for p in paths if not p.exists()]
        raise FileNotFoundError(f"Missing forgetting scores: {missing}")

    arrays = [np.load(p).astype(np.float32) for p in paths]
    np.save(
        DIRS["aggregated_scores"] / "forgetting_final.npy",
        aggregate_dense(arrays),
    )

print("Aggregated scores:", DIRS["aggregated_scores"])


## 8. Формирование subset

Для каждого активного метода создаются подмножества, содержащие 75%, 50% и 25% CIFAR-10 train set.


In [ ]:
def top_score_subset(
    scores: np.ndarray,
    fraction: float,
):
    n = int(round(len(scores) * fraction))

    safe = np.asarray(
        scores,
        dtype=np.float64,
    ).copy()

    safe[np.isnan(safe)] = -np.inf

    idx = np.argsort(safe)[-n:]

    return np.sort(
        idx.astype(np.int64)
    )

def random_subset_indices(
    n_total: int,
    fraction: float,
    seed: int,
):
    n = int(round(n_total * fraction))

    rng = np.random.RandomState(seed)

    return np.sort(
        rng.choice(
            n_total,
            size=n,
            replace=False,
        ).astype(np.int64)
    )

def subset_path(
    method: str,
    fraction: float,
    seed: int,
):
    pct = int(round(fraction * 100))
    return (
        DIRS["subsets"]
        / f"{method}_{pct:03d}_seed_{seed}.npy"
    )

def locate_archived_grand_seed_dir() -> Path:
    name = GRAND_ARCHIVE_DIR_NAME

    candidates = [
        PROJECT_BASE / MODE_NAME / "scores" / "grand" / name,
        PROJECT_ROOT / "scores" / "grand" / name,
        PROJECT_BASE / "production" / "scores" / "grand" / name,
        PROJECT_BASE / "production" / EXPERIMENT_NAMESPACE / "scores" / "grand" / name,
    ]

    for p in candidates:
        if p.exists():
            return p

    matches = [
        p for p in PROJECT_BASE.rglob(name)
        if p.is_dir()
        and (p / f"epoch_{GRAND_ACTIVE_SELECTION_EPOCH:03d}.npz").exists()
    ]

    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise RuntimeError(
            "Multiple archived GraNd directories found. Matches: "
            + ", ".join(str(p) for p in matches)
        )

    raise FileNotFoundError(
        "Archived GraNd directory was not found. Expected a directory named "
        f"{name!r} with epoch_{GRAND_ACTIVE_SELECTION_EPOCH:03d}.npz under {PROJECT_BASE}."
    )

def load_archived_grand_dense(epoch: int = GRAND_ACTIVE_SELECTION_EPOCH) -> np.ndarray:
    seed_dir = locate_archived_grand_seed_dir()
    path = seed_dir / f"epoch_{int(epoch):03d}.npz"

    if not path.exists():
        raise FileNotFoundError(f"Archived GraNd file not found: {path}")

    payload = np.load(path)

    if "grand" not in payload.files:
        raise KeyError(
            f"{path} has no 'grand' array. Available keys: {payload.files}"
        )

    grand = payload["grand"].astype(np.float32)

    if "indices" in payload.files:
        indices = payload["indices"].astype(np.int64)
    elif len(grand) == len(TRAIN_EVAL_DATASET):
        indices = np.arange(len(grand), dtype=np.int64)
    else:
        raise KeyError(
            f"{path} has no 'indices' array and only {len(grand)} scores; "
            f"expected {len(TRAIN_EVAL_DATASET)} for dense scores."
        )

    if len(indices) != len(grand):
        raise ValueError(
            f"GraNd indices/scores length mismatch: {len(indices)} vs {len(grand)}"
        )

    full = np.full(
        len(TRAIN_EVAL_DATASET),
        np.nan,
        dtype=np.float32,
    )
    full[indices] = grand

    n_finite = int(np.isfinite(full).sum())
    needed = int(round(len(TRAIN_EVAL_DATASET) * max(RETAIN_FRACTIONS)))
    if n_finite < needed:
        raise ValueError(
            f"Archived GraNd file has only {n_finite} finite scores, "
            f"but the largest retained subset needs {needed}."
        )

    aggregate_path = (
        DIRS["aggregated_scores"]
        / f"grand_epoch_{int(epoch):03d}.npz"
    )
    np.savez_compressed(
        aggregate_path,
        indices=indices,
        grand=grand,
    )

    print("[GraNd reuse]", path)
    print("[GraNd reuse] finite scores:", n_finite)
    print("[GraNd reuse] aggregate copy:", aggregate_path)

    return full

def load_selection_scores(method: str):
    if method == "el2n":
        path = (
            DIRS["aggregated_scores"]
            / f"el2n_epoch_{EL2N_ACTIVE_SELECTION_EPOCH:03d}.npy"
        )
        if not path.exists():
            raise FileNotFoundError(
                f"EL2N aggregate not found: {path}. "
                "Run the score-computation/aggregation cells or copy the "
                "production score artifacts from the baseline account."
            )
        return np.load(path)

    if method == "forgetting":
        path = (
            DIRS["aggregated_scores"]
            / "forgetting_final.npy"
        )
        if not path.exists():
            raise FileNotFoundError(
                f"Forgetting aggregate not found: {path}. "
                "Copy the complete scoring artifacts from the baseline account "
                "or run the aggregation cell."
            )
        return np.load(path)

    if method == "grand":
        return load_archived_grand_dense(
            GRAND_ACTIVE_SELECTION_EPOCH
        )

    if method == "random":
        return None

    raise ValueError(method)

selection_scores = {
    method: load_selection_scores(method)
    for method in ACTIVE_METHODS
}

rows = []

for eval_seed in EVAL_SEEDS:
    for method in ACTIVE_METHODS:
        for fraction in RETAIN_FRACTIONS:
            p = subset_path(
                method,
                fraction,
                eval_seed,
            )

            if method == "random":
                if not p.exists():
                    random_seed = 4426 + eval_seed * 424242
                    np.save(
                        p,
                        random_subset_indices(
                            len(TRAIN_AUG_DATASET),
                            fraction,
                            random_seed,
                        ),
                    )
            else:
                if not p.exists():
                    np.save(
                        p,
                        top_score_subset(
                            selection_scores[method],
                            fraction,
                        ),
                    )

            idx = np.load(p)

            rows.append({
                "method": method,
                "fraction": fraction,
                "seed": eval_seed,
                "n_examples": len(idx),
                "path": str(p),
            })

subset_table = pd.DataFrame(rows)
display(subset_table)


## 9. Retraining на выбранных subset

Каждая subset-модель стартует с новой случайной инициализации и использует тот же training protocol, что и baseline.


In [ ]:
subset_summaries = []

print("Training method(s) on this Colab:", ACTIVE_METHODS)
print("Active run plan:", ACTIVE_RUN_PLAN)
print("Subset run root:", DIRS["subset_runs"])
print("Hard step cap per run:", f"{SUBSET_TARGET_STEPS:,}")
print("Each run can finish earlier via plateau / final-stage early stopping.")

for eval_seed in EVAL_SEEDS:
    for method, fraction in ACTIVE_RUN_PLAN:
        pct = int(round(fraction * 100))

        indices = np.load(
            subset_path(method, fraction, eval_seed)
        )

        run_dir = (
            DIRS["subset_runs"]
            / f"{method}_{pct:03d}"
            / f"seed_{eval_seed}"
        )

        summary = train_fixed_steps(
            run_dir=run_dir,
            train_dataset=TRAIN_AUG_DATASET,
            test_dataset=TEST_DATASET,
            seed=eval_seed,
            method=method,
            fraction=fraction,
            subset_indices=indices,
            track_forgetting=False,
            save_score_checkpoints=False,
            target_steps=SUBSET_TARGET_STEPS,
        )

        subset_summaries.append(summary)
        update_live_results(summary)

pd.DataFrame(subset_summaries)


## 10. Сбор результатов

Готовые `summary.json` объединяются в общую таблицу; при нескольких seed дополнительно считаются mean и std.


In [ ]:
def collect_results_fast():
    rows = []

    for seed_dir in sorted(DIRS["scoring_runs"].glob("seed_*")):
        summary_path = seed_dir / "summary.json"
        if not summary_path.exists():
            continue
        with open(summary_path) as f:
            summary = json.load(f)
        if bool(summary.get("complete", False)):
            rows.append(summary)

    if DIRS["subset_runs"].exists():
        for summary_path in sorted(
            DIRS["subset_runs"].glob("*/*/summary.json")
        ):
            with open(summary_path) as f:
                summary = json.load(f)

            if bool(summary.get("complete", False)):
                rows.append(summary)

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    return (
        df.sort_values(["method", "fraction", "seed"])
        .reset_index(drop=True)
    )

all_runs = collect_results_fast()

all_runs_path = DIRS["results"] / "all_runs.csv"
all_runs.to_csv(all_runs_path, index=False)

if len(all_runs):
    agg = (
        all_runs
        .groupby(["method", "fraction"], as_index=False)
        .agg(
            n=("final_test_accuracy", "size"),
            final_test_accuracy_mean=("final_test_accuracy", "mean"),
            final_test_accuracy_std=("final_test_accuracy", "std"),
            best_test_accuracy_mean=("best_test_accuracy", "mean"),
            best_test_accuracy_std=("best_test_accuracy", "std"),
        )
    )

    agg.to_csv(
        DIRS["results"] / "aggregate.csv",
        index=False,
    )

    presentation = agg.copy()
    presentation["retained_data_%"] = (presentation["fraction"] * 100).astype(int)
    presentation["final_test_accuracy_%"] = presentation["final_test_accuracy_mean"] * 100
    presentation["final_test_accuracy_std_%"] = presentation["final_test_accuracy_std"] * 100
    presentation["best_test_accuracy_%"] = presentation["best_test_accuracy_mean"] * 100
    display(
        presentation[
            [
                "method",
                "retained_data_%",
                "n",
                "final_test_accuracy_%",
                "final_test_accuracy_std_%",
                "best_test_accuracy_%",
            ]
        ].sort_values(["retained_data_%", "method"], ascending=[False, True])
    )
else:
    print("No completed fast runs found yet.")

runtime_rows = []

runtime_paths = []
runtime_paths.extend(
    sorted(DIRS["scoring_runs"].glob("seed_*/metrics.csv"))
)
runtime_paths.extend(
    sorted(DIRS["subset_runs"].glob("*/*/metrics.csv"))
)

for metrics_path in runtime_paths:
    metrics = pd.read_csv(metrics_path)

    if "window_seconds" not in metrics.columns:
        continue

    runtime_rows.append({
        "run": str(metrics_path.parent.relative_to(PROJECT_ROOT)),
        "logged_windows": len(metrics),
        "seconds": float(metrics["window_seconds"].sum()),
        "hours": float(metrics["window_seconds"].sum()) / 3600.0,
    })

runtime_df = pd.DataFrame(runtime_rows)

if len(runtime_df):
    runtime_df.to_csv(
        DIRS["results"] / "runtime_summary.csv",
        index=False,
    )

    print(
        "Total logged runtime in included runs:",
        f"{runtime_df['hours'].sum():.2f} h",
    )

print("Results:", all_runs_path)


## 11. Графики

Следующие ячейки не запускают обучение и могут выполняться отдельно после завершения всех run.


In [ ]:
import os
import sys
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr

PROJECT_NAME = "data_diet_reproduction"
PLOT_MODE = "production"
PLOT_EXPERIMENT_NAMESPACE = "fast_bs512_plateau_v2"

PLOT_IN_COLAB = "google.colab" in sys.modules

if PLOT_IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PLOT_BASE = Path("/content/drive/MyDrive") / PROJECT_NAME
else:
    PLOT_BASE = Path.cwd() / PROJECT_NAME

PLOT_ROOT = PLOT_BASE / PLOT_MODE / PLOT_EXPERIMENT_NAMESPACE
PLOT_CONFIG_PATH = PLOT_ROOT / "configs" / "config_protocol.json"

with open(PLOT_CONFIG_PATH) as f:
    PLOT_CFG = json.load(f)

PLOT_RESULTS = PLOT_ROOT / "results"
PLOT_FIGURES = PLOT_ROOT / "figures"
PLOT_SCORES = PLOT_ROOT / "scores"
PLOT_AGG = PLOT_SCORES / "aggregated"
PLOT_DATA = PLOT_BASE / "data"
PLOT_SUBSET_RUNS = PLOT_ROOT / "runs" / "subsets_lr0008_v3"
PLOT_SCORING_RUNS = PLOT_ROOT / "runs" / "scoring"
PLOT_TOTAL_STEPS = int(PLOT_CFG["total_steps"])

PLOT_FIGURES.mkdir(parents=True, exist_ok=True)

print("Plot root:", PLOT_ROOT)
print("Subset run root:", PLOT_SUBSET_RUNS)
print("Results:", PLOT_RESULTS)
print(json.dumps(PLOT_CFG, indent=2))


In [ ]:
def _save_show(
    fig,
    path: Path,
):
    fig.tight_layout()
    fig.savefig(
        path,
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

def load_all_runs_plot():
    path = PLOT_RESULTS / "all_runs.csv"

    if path.exists():
        return pd.read_csv(path)

    rows = []

    for summary_path in sorted(
        PLOT_SCORING_RUNS.glob(
            "seed_*/summary.json"
        )
    ):
        with open(summary_path) as f:
            rows.append(
                json.load(f)
            )

    for summary_path in sorted(
        PLOT_SUBSET_RUNS.glob(
            "*/*/summary.json"
        )
    ):
        with open(summary_path) as f:
            rows.append(
                json.load(f)
            )

    if not rows:
        raise FileNotFoundError(
            "No completed run summaries found."
        )

    df = pd.DataFrame(rows)
    df.to_csv(
        path,
        index=False,
    )
    return df

def plot_pruning_curve():
    df = load_all_runs_plot()

    fig, ax = plt.subplots(
        figsize=(7, 5)
    )

    full = df[
        df["method"] == "full"
    ]

    if len(full):
        full_mean = (
            full["final_test_accuracy"]
            .mean()
        )
        ax.axhline(
            full_mean * 100,
            linestyle="--",
            label="Full (100%)",
        )

    for method in [
        "el2n",
        "forgetting",
        "grand",
        "random",
    ]:
        sub = df[
            df["method"] == method
        ]

        if not len(sub):
            continue

        stats = (
            sub.groupby(
                "fraction"
            )["final_test_accuracy"]
            .agg(
                ["mean", "std", "count"]
            )
            .reset_index()
            .sort_values(
                "fraction"
            )
        )

        yerr = np.nan_to_num(
            stats["std"].to_numpy(),
            nan=0.0,
        )

        ax.errorbar(
            stats["fraction"] * 100,
            stats["mean"] * 100,
            yerr=yerr * 100,
            marker="o",
            capsize=3,
            label=method,
        )

    ax.set_xlabel(
        "Training data retained (%)"
    )
    ax.set_ylabel(
        "Final test accuracy (%)"
    )
    ax.set_title(
        "Accuracy vs retained training data"
    )
    ax.grid(
        True,
        alpha=0.25,
    )
    ax.legend()

    _save_show(
        fig,
        PLOT_FIGURES
        / "pruning_curve.png",
    )

def _metrics_path_for(
    method: str,
    fraction: float,
    seed: int,
):
    if method == "full":
        return (
            PLOT_SCORING_RUNS
            / f"seed_{seed}"
            / "metrics.csv"
        )

    pct = int(round(
        fraction * 100
    ))

    return (
        PLOT_SUBSET_RUNS
        / f"{method}_{pct:03d}"
        / f"seed_{seed}"
        / "metrics.csv"
    )

def plot_learning_curves_50pct():
    scoring_seed = (
        PLOT_CFG["scoring_seeds"][0]
    )
    eval_seed = (
        PLOT_CFG["eval_seeds"][0]
    )

    fig, ax = plt.subplots(
        figsize=(7, 5)
    )

    curves = [
        (
            "full", 1.0,
            scoring_seed,
            "Full",
        ),
        (
            "el2n", 0.5,
            eval_seed,
            "EL2N 50%",
        ),
        (
            "forgetting", 0.5,
            eval_seed,
            "Forgetting 50%",
        ),
        (
            "grand", 0.5,
            eval_seed,
            "GraNd@20 50%",
        ),
        (
            "random", 0.5,
            eval_seed,
            "Random 50%",
        ),
    ]

    for (
        method,
        fraction,
        seed,
        label,
    ) in curves:
        path = _metrics_path_for(
            method,
            fraction,
            seed,
        )

        if not path.exists():
            continue

        m = pd.read_csv(path)

        ax.plot(
            m["equiv_epoch"],
            m["test_accuracy"] * 100,
            marker="o",
            label=label,
        )

    ax.set_xlabel(
        "Full-data-equivalent epoch"
    )
    ax.set_ylabel(
        "Test accuracy (%)"
    )
    ax.set_title(
        "Learning curves at 50% retained data"
    )
    ax.grid(
        True,
        alpha=0.25,
    )
    ax.legend()

    _save_show(
        fig,
        PLOT_FIGURES
        / "learning_curves_50pct.png",
    )

def plot_el2n_distribution():
    epoch = int(
        PLOT_CFG[
            "el2n_selection_epoch"
        ]
    )

    scores = np.load(
        PLOT_AGG
        / f"el2n_epoch_{epoch:03d}.npy"
    )

    scores = scores[
        np.isfinite(scores)
    ]

    fig, ax = plt.subplots(
        figsize=(7, 5)
    )
    ax.hist(
        scores,
        bins=60,
    )
    ax.set_xlabel(
        "EL2N score"
    )
    ax.set_ylabel(
        "Number of training examples"
    )
    ax.set_title(
        f"EL2N distribution at epoch {epoch}"
    )

    _save_show(
        fig,
        PLOT_FIGURES
        / "el2n_distribution.png",
    )

def plot_forgetting_distribution():
    scores = np.load(
        PLOT_AGG / "forgetting_final.npy"
    )

    finite = scores[
        np.isfinite(scores)
    ]
    never = int(
        (~np.isfinite(scores)).sum()
    )

    fig, ax = plt.subplots(
        figsize=(7, 5)
    )

    if len(finite):
        bins = (
            np.arange(
                finite.max() + 2
            )
            - 0.5
        )
        ax.hist(
            finite,
            bins=bins,
        )

    ax.set_xlabel(
        "Forgetting count"
    )
    ax.set_ylabel(
        "Number of training examples"
    )
    ax.set_title(
        f"Forgetting distribution at final adaptive state | "
        f"never correct: {never}"
    )

    _save_show(
        fig,
        PLOT_FIGURES
        / "forgetting_distribution.png",
    )

def plot_grand_distribution():
    epoch = int(PLOT_CFG.get("grand_selection_epoch", 20))
    path = PLOT_AGG / f"grand_epoch_{epoch:03d}.npz"

    if not path.exists():
        print("GraNd aggregate not found:", path)
        return

    d = np.load(path)
    scores = d["grand"]
    scores = scores[np.isfinite(scores)]

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.hist(scores, bins=60)
    ax.set_xlabel("GraNd score")
    ax.set_ylabel("Number of scored examples")
    ax.set_title(f"GraNd distribution at epoch {epoch}")

    _save_show(fig, PLOT_FIGURES / "grand_distribution.png")

def plot_el2n_rank_stability():
    epochs = [
        int(e)
        for e in PLOT_CFG[
            "score_epochs"
        ]
    ]

    target_epoch = int(
        PLOT_CFG[
            "el2n_selection_epoch"
        ]
    )

    target = np.load(
        PLOT_AGG
        / f"el2n_epoch_{target_epoch:03d}.npy"
    )

    xs = []
    ys = []

    for epoch in epochs:
        path = (
            PLOT_AGG
            / f"el2n_epoch_{epoch:03d}.npy"
        )

        if not path.exists():
            continue

        scores = np.load(path)

        mask = (
            np.isfinite(scores)
            & np.isfinite(target)
        )

        rho = spearmanr(
            scores[mask],
            target[mask],
        ).statistic

        xs.append(epoch)
        ys.append(rho)

    fig, ax = plt.subplots(
        figsize=(7, 5)
    )

    ax.plot(
        xs,
        ys,
        marker="o",
    )
    ax.set_xlabel(
        "EL2N scoring epoch"
    )
    ax.set_ylabel(
        f"Spearman correlation with EL2N@{target_epoch}"
    )
    ax.set_ylim(
        -0.05,
        1.05,
    )
    ax.set_title(
        "EL2N ranking stability during early training"
    )
    ax.grid(
        True,
        alpha=0.25,
    )

    _save_show(
        fig,
        PLOT_FIGURES
        / "el2n_rank_stability.png",
    )

def _rankable_values(values):
    values = np.asarray(values, dtype=np.float64).copy()
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return values
    lo = finite.min()
    hi = finite.max()
    span = max(hi - lo, 1.0)
    values[np.isposinf(values)] = hi + span
    values[np.isneginf(values)] = lo - span
    return values

def plot_score_correlation_matrix():
    el2n_epoch = int(
        PLOT_CFG[
            "el2n_selection_epoch"
        ]
    )
    el2n = np.load(
        PLOT_AGG
        / f"el2n_epoch_{el2n_epoch:03d}.npy"
    )

    forgetting = np.load(
        PLOT_AGG / "forgetting_final.npy"
    )

    score_dict = {
        f"EL2N@{el2n_epoch}": (
            np.arange(len(el2n)),
            el2n,
        ),
        "Forget@final": (
            np.arange(len(forgetting)),
            forgetting,
        ),
    }

    grand_epoch = int(PLOT_CFG.get("grand_selection_epoch", 20))
    grand_path = PLOT_AGG / f"grand_epoch_{grand_epoch:03d}.npz"

    if grand_path.exists():
        gd = np.load(grand_path)
        score_dict[f"GraNd@{grand_epoch}"] = (
            gd["indices"].astype(np.int64),
            gd["grand"].astype(np.float64),
        )

    names = list(score_dict)

    corr = np.full(
        (len(names), len(names)),
        np.nan,
        dtype=float,
    )

    for i, name_i in enumerate(names):
        idx_i, scores_i = (
            score_dict[name_i]
        )
        scores_i = _rankable_values(scores_i)

        map_i = {
            int(k): float(v)
            for k, v in zip(
                idx_i,
                scores_i,
            )
            if not np.isnan(v)
        }

        for j, name_j in enumerate(names):
            idx_j, scores_j = (
                score_dict[name_j]
            )
            scores_j = _rankable_values(scores_j)

            map_j = {
                int(k): float(v)
                for k, v in zip(
                    idx_j,
                    scores_j,
                )
                if not np.isnan(v)
            }

            common = sorted(
                set(map_i)
                .intersection(map_j)
            )

            if len(common) < 3:
                continue

            a = np.asarray([
                map_i[k]
                for k in common
            ])
            b = np.asarray([
                map_j[k]
                for k in common
            ])

            corr[i, j] = spearmanr(
                a, b
            ).statistic

    fig, ax = plt.subplots(
        figsize=(6, 5)
    )

    im = ax.imshow(
        corr,
        vmin=-1,
        vmax=1,
    )

    ax.set_xticks(
        range(len(names)),
        names,
        rotation=30,
        ha="right",
    )
    ax.set_yticks(
        range(len(names)),
        names,
    )
    ax.set_title(
        "Spearman correlations between data scores"
    )

    for i in range(len(names)):
        for j in range(len(names)):
            if np.isfinite(
                corr[i, j]
            ):
                ax.text(
                    j,
                    i,
                    f"{corr[i, j]:.2f}",
                    ha="center",
                    va="center",
                )

    fig.colorbar(
        im,
        ax=ax,
    )

    _save_show(
        fig,
        PLOT_FIGURES
        / "score_correlation_matrix.png",
    )

def _load_sorted_raw_train():
    from torchvision.datasets import CIFAR10

    base = CIFAR10(
        root=str(PLOT_DATA),
        train=True,
        download=True,
        transform=None,
    )

    targets = np.asarray(
        base.targets
    )
    order = np.argsort(
        targets,
        kind="stable",
    )

    return base, order

def plot_el2n_extremes_by_class(
    n_per_class: int = 2,
):
    epoch = int(
        PLOT_CFG[
            "el2n_selection_epoch"
        ]
    )

    scores = np.load(
        PLOT_AGG
        / f"el2n_epoch_{epoch:03d}.npy"
    )

    base, order = (
        _load_sorted_raw_train()
    )

    labels_sorted = (
        np.asarray(base.targets)[order]
    )

    class_names = [
        "airplane", "automobile",
        "bird", "cat", "deer",
        "dog", "frog", "horse",
        "ship", "truck",
    ]

    for mode in [
        "lowest",
        "highest",
    ]:
        chosen = []

        for cls in range(10):
            cls_idx = np.flatnonzero(
                labels_sorted == cls
            )

            cls_scores = scores[
                cls_idx
            ]

            finite_mask = np.isfinite(
                cls_scores
            )

            cls_idx = cls_idx[
                finite_mask
            ]
            cls_scores = cls_scores[
                finite_mask
            ]

            order_local = np.argsort(
                cls_scores
            )

            if mode == "highest":
                order_local = (
                    order_local[::-1]
                )

            for k in order_local[
                :n_per_class
            ]:
                chosen.append(
                    int(cls_idx[k])
                )

        ncols = 5
        nrows = math.ceil(
            len(chosen) / ncols
        )

        fig = plt.figure(
            figsize=(
                12,
                2.3 * nrows,
            )
        )

        for j, sorted_idx in enumerate(
            chosen
        ):
            ax = fig.add_subplot(
                nrows,
                ncols,
                j + 1,
            )

            base_idx = int(
                order[sorted_idx]
            )

            img, label = base[
                base_idx
            ]

            ax.imshow(img)
            ax.set_title(
                f"{class_names[label]}\n"
                f"EL2N={scores[sorted_idx]:.3f}"
            )
            ax.axis("off")

        fig.suptitle(
            f"{mode.capitalize()} EL2N examples by class "
            f"— epoch {epoch}"
        )

        _save_show(
            fig,
            PLOT_FIGURES
            / f"el2n_{mode}_by_class.png",
        )

def plot_score_seed_stability():
    seeds = list(
        PLOT_CFG.get(
            "scoring_seeds",
            [],
        )
    )

    epoch = int(
        PLOT_CFG[
            "el2n_selection_epoch"
        ]
    )

    if len(seeds) < 2:
        print(
            "Seed stability plot skipped: "
            "only one scoring seed."
        )
        return

    arrays = []

    for seed in seeds:
        path = (
            PLOT_SCORES
            / "el2n"
            / f"seed_{seed}"
            / f"epoch_{epoch:03d}.npz"
        )

        d = np.load(path)

        arr = np.full(
            50000,
            np.nan,
            dtype=np.float32,
        )

        arr[
            d["indices"].astype(np.int64)
        ] = d[
            "el2n"
        ].astype(np.float32)

        arrays.append(arr)

    corr = np.full(
        (len(seeds), len(seeds)),
        np.nan,
    )

    for i in range(len(seeds)):
        for j in range(len(seeds)):
            mask = (
                np.isfinite(arrays[i])
                & np.isfinite(arrays[j])
            )

            corr[i, j] = spearmanr(
                arrays[i][mask],
                arrays[j][mask],
            ).statistic

    fig, ax = plt.subplots(
        figsize=(6, 5)
    )

    im = ax.imshow(
        corr,
        vmin=-1,
        vmax=1,
    )

    labels = [
        f"seed {s}"
        for s in seeds
    ]

    ax.set_xticks(
        range(len(seeds)),
        labels,
        rotation=30,
        ha="right",
    )
    ax.set_yticks(
        range(len(seeds)),
        labels,
    )

    ax.set_title(
        f"EL2N seed-to-seed rank correlation @ epoch {epoch}"
    )

    for i in range(len(seeds)):
        for j in range(len(seeds)):
            ax.text(
                j,
                i,
                f"{corr[i, j]:.2f}",
                ha="center",
                va="center",
            )

    fig.colorbar(
        im,
        ax=ax,
    )

    _save_show(
        fig,
        PLOT_FIGURES
        / "el2n_seed_stability.png",
    )


In [ ]:
plot_pruning_curve()
plot_learning_curves_50pct()
plot_el2n_distribution()
plot_forgetting_distribution()
plot_grand_distribution()
plot_el2n_rank_stability()
plot_score_correlation_matrix()
plot_el2n_extremes_by_class(n_per_class=2)
plot_score_seed_stability()

print("Figures saved to:", PLOT_FIGURES)
